# Multiagentes no Kaggle: legislação, Analista e Validador

O **Analista (Qwen3 8B)** escolhe a consulta, usa a busca e redige. O **Validador (Gemma 3 12B)** verifica a explicação contra as mesmas fontes e pode pedir **uma revisão**. LangGraph coordena o fluxo e os guardrails controlam a liberação.

A execução principal usa o **corpus de legislação real selecionado no pacote**, com busca lexical (`modo=corpus`) ou embeddings BGE-M3 (`modo=local`), conforme o manifesto. Não consulta PostgreSQL, não utiliza o índice de jurisprudência e não confirma a vigência das normas.

Este notebook está sem execuções e sem resultados preenchidos. No Kaggle, adicione o dataset **privado**, selecione **GPU T4**, ligue **Internet** e execute **Run All**. A configuração abaixo já habilita a execução. Não são necessárias chaves de API, credenciais de banco ou token do Kaggle dentro do notebook.

No modo local, a primeira indexação baixa BGE-M3 e prepara o cache antes do atendimento. Os embeddings usam CPU por padrão para preservar a memória de GPU de Qwen/Gemma.

Os modelos são baixados um após o outro e descarregados após cada pedido: não precisam permanecer juntos na memória da GPU. Downloads, disponibilidade de GPU e duração da sessão dependem do ambiente.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import importlib
import json
import os
import platform
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import time
import urllib.request
import uuid
import zipfile

EXECUTAR = True
EXECUTAR_DEMO_REVISAO = False  # Opcional: outra execução, com material fictício.
DIRETORIO_DATASET = None      # Ex.: "/kaggle/input/nome-do-dataset", se houver mais de um pacote.
PERGUNTA = "Quais são as regras gerais sobre férias segundo as fontes fornecidas?"
DISPOSITIVO_EMBEDDINGS = "cpu"  # Troque por "cuda" para indexar na GPU da sessão.
BATCH_SIZE_EMBEDDINGS = 2     # Reduz o pico de memória nos artigos longos.
CACHE_EMBEDDINGS = Path("/kaggle/working/dados/output/buscador_local")

OLLAMA_VERSAO = "0.40.16"
LANGGRAPH_VERSAO = "1.2.22"
MODELOS = ["qwen3:8b-q8_0", "gemma3:12b"]
URL_OLLAMA = "http://127.0.0.1:11434"
WORKING = Path("/kaggle/working")
DIRETORIO_SAIDA = WORKING / "multiagentes"
ESTADO = None
ESTADO_DEMO = None
ERRO_EXECUCAO = None
AMBIENTE = {}

print("Execução habilitada:", EXECUTAR)
print("Demonstração sintética adicional:", EXECUTAR_DEMO_REVISAO)


## Conferir o ambiente

O resumo mostra Python, pip e GPU, sem listar variáveis de ambiente ou credenciais. Os próximos passos verificam o pacote antes de extrair e importar seu código.


In [ ]:
if EXECUTAR:
    if not Path("/kaggle/input").is_dir() or not WORKING.is_dir():
        raise RuntimeError("Execute este notebook no Kaggle, após adicionar o dataset do pacote.")
    candidatos_smi = [shutil.which("nvidia-smi"), "/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi"]
    executavel_smi = next((p for p in candidatos_smi if p and Path(p).is_file()), None)
    gpu = subprocess.run(
        [executavel_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
        check=False, capture_output=True, text=True,
    ) if executavel_smi else None
    diagnostico_gpu = {
        "nvidia_smi": executavel_smi,
        "dispositivos_nvidia": sorted(p.name for p in Path("/dev").glob("nvidia*")),
        "retorno_smi": gpu.returncode if gpu else None,
        "saida_smi": gpu.stdout.strip() if gpu else None,
        "erro_smi": gpu.stderr.strip() if gpu else None,
    }
    print(json.dumps(diagnostico_gpu, ensure_ascii=False, indent=2))
    if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
        raise RuntimeError("GPU NVIDIA indisponível na sessão; consulte o diagnóstico acima.")
    pip_info = subprocess.run(
        [sys.executable, "-m", "pip", "--version"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    AMBIENTE.update(python=platform.python_version(), pip=pip_info, gpu=gpu.stdout.strip(), diagnostico_gpu=diagnostico_gpu)
    print(json.dumps(AMBIENTE, ensure_ascii=False, indent=2))


In [ ]:
def sha256_arquivo(caminho):
    digest = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()


def conferir_descricao(descricao, tamanho, digest, nome):
    if not isinstance(descricao, dict):
        raise RuntimeError(f"Descrição ausente no manifesto: {nome}")
    if type(descricao.get("bytes")) is not int or descricao["bytes"] < 0:
        raise RuntimeError(f"Tamanho inválido no manifesto: {nome}")
    esperado = descricao.get("sha256")
    if not isinstance(esperado, str) or not re.fullmatch(r"[0-9a-f]{64}", esperado):
        raise RuntimeError(f"SHA-256 inválido no manifesto: {nome}")
    if tamanho != descricao["bytes"] or digest != esperado:
        raise RuntimeError(f"Integridade divergente: {nome}. Reenvie o pacote completo.")


if EXECUTAR:
    if DIRETORIO_DATASET is not None:
        candidatos = [Path(DIRETORIO_DATASET) / "manifesto_pacote.json"]
    else:
        candidatos = [
            p for p in Path("/kaggle/input").rglob("manifesto_pacote.json")
            if any((p.parent / nome).is_file() for nome in ("corpus_legislacao.json", "legislacao_artigos_completos.json"))
            and ((p.parent / "runtime.zip").is_file() or (p.parent / "runtime").is_dir())
        ]
    if len(candidatos) != 1:
        raise RuntimeError(
            "Adicione um dataset com runtime.zip (ou pasta runtime/), o JSON de legislação e manifesto_pacote.json. "
            "Se houver mais de um pacote, preencha DIRETORIO_DATASET na configuração."
        )
    MANIFESTO_PATH = candidatos[0]
    if not MANIFESTO_PATH.is_file():
        raise RuntimeError("O diretório informado não contém manifesto_pacote.json.")
    PACOTE_DIR = MANIFESTO_PATH.parent.resolve()
    MANIFESTO = json.loads(MANIFESTO_PATH.read_text(encoding="utf-8"))
    if MANIFESTO.get("versao_pacote") != "multiagentes-kaggle-v1" or MANIFESTO.get("modo") not in ("corpus", "local"):
        raise RuntimeError("Versão ou modo do pacote incompatível com este notebook.")
    descricoes = MANIFESTO.get("arquivos")
    if not isinstance(descricoes, dict):
        raise RuntimeError("O manifesto não contém o inventário de arquivos.")
    MODO_BUSCA = MANIFESTO["modo"]
    NOME_CORPUS = "legislacao_artigos_completos.json" if MODO_BUSCA == "local" else "corpus_legislacao.json"
    CORPUS_PATH = PACOTE_DIR / NOME_CORPUS
    if not CORPUS_PATH.is_file() or CORPUS_PATH.resolve().parent != PACOTE_DIR:
        raise RuntimeError("Arquivo do pacote ausente ou inválido: " + NOME_CORPUS)
    conferir_descricao(
        descricoes.get(NOME_CORPUS), CORPUS_PATH.stat().st_size,
        sha256_arquivo(CORPUS_PATH), NOME_CORPUS,
    )
    RUNTIME_ZIP = PACOTE_DIR / "runtime.zip"
    RUNTIME_DIR_ORIGEM = PACOTE_DIR / "runtime"
    if RUNTIME_ZIP.is_file():
        if RUNTIME_ZIP.resolve().parent != PACOTE_DIR:
            raise RuntimeError("Arquivo do pacote ausente ou inválido: runtime.zip")
        conferir_descricao(
            descricoes.get("runtime.zip"), RUNTIME_ZIP.stat().st_size,
            sha256_arquivo(RUNTIME_ZIP), "runtime.zip",
        )
        RUNTIME_ORIGEM = RUNTIME_ZIP
    elif RUNTIME_DIR_ORIGEM.is_dir() and RUNTIME_DIR_ORIGEM.resolve().parent == PACOTE_DIR:
        RUNTIME_ORIGEM = RUNTIME_DIR_ORIGEM
    else:
        raise RuntimeError("Pacote sem runtime.zip ou diretório runtime/.")
    print("Pacote verificado:", PACOTE_DIR.name)
    print("Tema:", MANIFESTO.get("tema", "não informado"))
    print("Modo de busca:", MODO_BUSCA)
    print("Corpus SHA-256:", descricoes[NOME_CORPUS]["sha256"])


In [ ]:
def caminho_runtime_permitido(nome):
    if not isinstance(nome, str) or not nome or "\\" in nome or "\x00" in nome:
        return False
    partes = PurePosixPath(nome)
    if partes.is_absolute() or ".." in partes.parts or str(partes) != nome:
        return False
    return nome in ("requirements-multiagentes.txt", "requirements-buscador-local.txt") or (
        len(partes.parts) >= 2 and partes.parts[0] == "src" and partes.suffix == ".py"
    )


if EXECUTAR:
    inventario = MANIFESTO.get("runtime", {}).get("arquivos")
    if not isinstance(inventario, dict) or not inventario:
        raise RuntimeError("O manifesto não contém o inventário interno de runtime.zip.")
    if not all(caminho_runtime_permitido(nome) for nome in inventario):
        raise RuntimeError("Inventário com caminho ou tipo de arquivo não permitido.")
    if "requirements-multiagentes.txt" not in inventario:
        raise RuntimeError("O runtime precisa conter requirements-multiagentes.txt.")
    if MODO_BUSCA == "local" and "requirements-buscador-local.txt" not in inventario:
        raise RuntimeError("O runtime local precisa conter requirements-buscador-local.txt.")
    conteudos_verificados = {}
    if RUNTIME_ORIGEM.is_file():
        with zipfile.ZipFile(RUNTIME_ORIGEM) as pacote_zip:
            entradas = pacote_zip.infolist()
            nomes = [info.filename for info in entradas]
            if len(nomes) != len(set(nomes)) or set(nomes) != set(inventario):
                raise RuntimeError("Entradas duplicadas ou inventário divergente no ZIP.")
            if sum(info.file_size for info in entradas) > 16 * 1024 * 1024:
                raise RuntimeError("Runtime acima do limite de extração deste notebook.")
            for info in entradas:
                modo_arquivo = info.external_attr >> 16
                if (
                    not caminho_runtime_permitido(info.filename)
                    or info.is_dir()
                    or stat.S_ISLNK(modo_arquivo)
                    or stat.S_IFMT(modo_arquivo) not in (0, stat.S_IFREG)
                    or info.flag_bits & 1
                    or info.file_size > 2 * 1024 * 1024
                ):
                    raise RuntimeError("Entrada insegura ou tipo de arquivo não permitido no ZIP.")
                with pacote_zip.open(info) as arquivo:
                    conteudo = arquivo.read(info.file_size + 1)
                conferir_descricao(
                    inventario[info.filename], len(conteudo),
                    hashlib.sha256(conteudo).hexdigest(), info.filename,
                )
                conteudos_verificados[info.filename] = conteudo
    else:
        base_runtime = RUNTIME_ORIGEM.resolve()
        total_bytes = 0
        for entrada in sorted(RUNTIME_ORIGEM.rglob("*")):
            if entrada.is_symlink():
                raise RuntimeError("Atalho simbólico não permitido em runtime/.")
            if entrada.is_dir():
                continue
            if not entrada.is_file() or base_runtime not in entrada.resolve().parents:
                raise RuntimeError("Entrada insegura em runtime/.")
            rel = entrada.relative_to(RUNTIME_ORIGEM).as_posix()
            if not caminho_runtime_permitido(rel) or rel not in inventario:
                raise RuntimeError(f"Arquivo não permitido ou fora do inventário: {rel}")
            tamanho = entrada.stat().st_size
            if tamanho > 2 * 1024 * 1024:
                raise RuntimeError(f"Arquivo acima do limite em runtime/: {rel}")
            total_bytes += tamanho
            if total_bytes > 16 * 1024 * 1024:
                raise RuntimeError("Runtime acima do limite de extração deste notebook.")
            conteudo = entrada.read_bytes()
            conferir_descricao(
                inventario[rel], len(conteudo),
                hashlib.sha256(conteudo).hexdigest(), rel,
            )
            conteudos_verificados[rel] = conteudo
        if set(conteudos_verificados) != set(inventario):
            raise RuntimeError("Inventário divergente na pasta runtime/.")
    # Só cria arquivos após validar todas as entradas. Não usa extractall.
    RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="lexia_runtime_", dir=WORKING))
    for nome, conteudo in conteudos_verificados.items():
        destino = RUNTIME_DIR.joinpath(*PurePosixPath(nome).parts)
        if RUNTIME_DIR.resolve() not in destino.resolve().parents:
            raise RuntimeError("Caminho fora do diretório reservado ao runtime.")
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(conteudo)
    print("Runtime extraído após validação:", len(conteudos_verificados), "arquivos.")


In [ ]:
if EXECUTAR:
    requisitos = (RUNTIME_DIR / "requirements-multiagentes.txt").read_text(encoding="utf-8")
    if not re.search(r"(?m)^langgraph==1\.2\.22\s*$", requisitos):
        raise RuntimeError("A versão de LangGraph do pacote diverge da versão deste notebook.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
         f"langgraph=={LANGGRAPH_VERSAO}"],
        check=True,
    )
    if version("langgraph") != LANGGRAPH_VERSAO:
        raise RuntimeError("Não foi possível instalar a versão esperada de LangGraph.")
    if MODO_BUSCA == "local":
        requisitos_local = (RUNTIME_DIR / "requirements-buscador-local.txt").read_text(encoding="utf-8")
        if not re.search(r"(?m)^sentence-transformers>=3\.4,<6\s*$", requisitos_local):
            raise RuntimeError("Requisitos do buscador local incompatíveis com este notebook.")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
             "sentence-transformers>=3.4,<6"],
            check=True,
        )
        AMBIENTE["sentence_transformers"] = version("sentence-transformers")
    # Rerodar o notebook importa novamente apenas o código verificado deste pacote.
    for nome in list(sys.modules):
        if nome == "src" or nome.startswith("src."):
            del sys.modules[nome]
    sys.path.insert(0, str(RUNTIME_DIR))
    importlib.invalidate_caches()
    AMBIENTE["langgraph"] = version("langgraph")
    print("LangGraph:", AMBIENTE["langgraph"])


## Preparar Ollama e os dois modelos

O instalador oficial recebe a versão fixa usada pelo projeto. O servidor escuta apenas em **127.0.0.1:11434**. Os logs ficam em **/kaggle/working**.

A configuração limita a um modelo carregado por vez. O cliente do protótipo envia **keep_alive=0 em cada pedido**, descarregando o modelo ao concluir. Baixar os dois modelos usa disco; alternar entre eles pode aumentar a duração da execução.


In [ ]:
def api_ollama(endpoint, corpo=None, timeout=15):
    dados = None if corpo is None else json.dumps(corpo).encode("utf-8")
    pedido = urllib.request.Request(
        URL_OLLAMA + endpoint, data=dados,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(pedido, timeout=timeout) as resposta:
        return json.load(resposta)


if EXECUTAR:
    try:
        servidor_existente = api_ollama("/api/version")
    except (OSError, ValueError):
        servidor_existente = None
    if servidor_existente is not None and servidor_existente.get("version") != OLLAMA_VERSAO:
        raise RuntimeError("Há outro Ollama ativo. Reinicie a sessão antes de executar este pacote.")
    if servidor_existente is None:
        instalado = shutil.which("ollama")
        versao_cli = (
            subprocess.run(["ollama", "--version"], check=False, capture_output=True, text=True)
            if instalado else None
        )
        cli_compativel = versao_cli is not None and re.search(
            r"(?<![\d.])" + re.escape(OLLAMA_VERSAO) + r"(?![\d.])",
            versao_cli.stdout + versao_cli.stderr,
        )
        if not cli_compativel:
            instalador = WORKING / "instalar_ollama.sh"
            with urllib.request.urlopen("https://ollama.com/install.sh", timeout=60) as resposta:
                instalador.write_bytes(resposta.read())
            print("Instalando Ollama", OLLAMA_VERSAO, "— detalhes em ollama_instalacao.log.")
            with (WORKING / "ollama_instalacao.log").open("ab") as log:
                if not shutil.which("zstd") and shutil.which("apt-get"):
                    subprocess.run(
                        ["apt-get", "install", "-y", "-q", "zstd"],
                        check=True, stdout=log, stderr=log,
                    )
                subprocess.run(
                    ["sh", str(instalador)],
                    env={**os.environ, "OLLAMA_VERSION": OLLAMA_VERSAO},
                    check=True, stdout=log, stderr=log,
                )
        ambiente_servidor = {
            **os.environ,
            "OLLAMA_HOST": "127.0.0.1:11434",
            "OLLAMA_KEEP_ALIVE": "0",
            "OLLAMA_MAX_LOADED_MODELS": "1",
            "OLLAMA_NUM_PARALLEL": "1",
        }
        with (WORKING / "ollama.log").open("ab") as log:
            PROCESSO_OLLAMA = subprocess.Popen(
                ["ollama", "serve"], env=ambiente_servidor, stdout=log, stderr=log,
            )
        for tentativa in range(90):
            try:
                servidor_existente = api_ollama("/api/version", timeout=3)
                break
            except (OSError, ValueError):
                if PROCESSO_OLLAMA.poll() is not None:
                    raise RuntimeError("Ollama não iniciou. Consulte /kaggle/working/ollama.log.") from None
                time.sleep(2)
        else:
            raise RuntimeError("Ollama não respondeu. Consulte /kaggle/working/ollama.log.")
    if servidor_existente.get("version") != OLLAMA_VERSAO:
        raise RuntimeError("A versão do servidor Ollama diverge da versão configurada.")
    # Descarrega possíveis resíduos de uma execução anterior da própria sessão.
    for carregado in api_ollama("/api/ps").get("models", []):
        api_ollama("/api/generate", {"model": carregado["name"], "keep_alive": 0}, timeout=60)
    for indice, modelo in enumerate(MODELOS, 1):
        print(f"Preparando modelo {indice}/{len(MODELOS)}: {modelo}", flush=True)
        with (WORKING / "ollama_modelos.log").open("ab") as log:
            subprocess.run(["ollama", "pull", modelo], check=True, stdout=log, stderr=log)
    modelos_instalados = [
        {"nome": m.get("name"), "digest": m.get("digest"), "bytes": m.get("size"),
         "quantizacao": (m.get("details") or {}).get("quantization_level")}
        for m in api_ollama("/api/tags").get("models", [])
        if m.get("name") in MODELOS
    ]
    AMBIENTE.update(
        modelos_instalados=modelos_instalados,
        ollama=api_ollama("/api/version").get("version"),
        modelos=MODELOS,
        keep_alive_por_pedido=0,
        corpus_sha256=MANIFESTO["arquivos"][NOME_CORPUS]["sha256"],
        runtime_sha256=MANIFESTO["arquivos"]["runtime.zip"]["sha256"],
    )
    print(json.dumps(AMBIENTE, ensure_ascii=False, indent=2))


## Executar o atendimento com a legislação do pacote

O caso principal faz uma busca, redação, validação e, quando necessária, uma revisão. Uma revisão pode surgir naturalmente; ela não é forçada neste modo.

No modo local, a indexação semântica e a conferência do cache são preparadas antes do atendimento. O custo inicial pode ser alto na CPU; novas perguntas na mesma sessão reutilizam os embeddings dos artigos. A GPU continua dedicada aos dois modelos de linguagem.

As fontes permanecem congeladas durante o ciclo. Se a evidência for insuficiente, houver reprovação persistente ou ocorrer uma falha técnica, o status e a resposta final mostram esse resultado.


In [ ]:
if EXECUTAR:
    from src.multiagentes.cliente_llm import ClienteOllama
    from src.multiagentes.ferramentas import BuscaCorpus, BuscaSintetica
    from src.multiagentes.buscador_local import BuscadorLocal
    from src.multiagentes.orquestrador import ConfiguracaoSistema, SistemaMultiagentes
    from src.multiagentes.registro import salvar_execucao

    DIRETORIO_SAIDA.mkdir(parents=True, exist_ok=True, mode=0o700)
    CONFIG = ConfiguracaoSistema(
        timeout_segundos=600, tempo_maximo_segundos=3600,
        modelo_analista=MODELOS[0], modelo_validador=MODELOS[1],
    )
    cliente = ClienteOllama(URL_OLLAMA, timeout=600)
    busca_corpus = (
        BuscadorLocal(CORPUS_PATH, cache_dir=CACHE_EMBEDDINGS, device=DISPOSITIVO_EMBEDDINGS,
                      batch_size=BATCH_SIZE_EMBEDDINGS)
        if MODO_BUSCA == "local" else BuscaCorpus(CORPUS_PATH)
    )
    try:
        if MODO_BUSCA == "local":
            print("Preparando índice BGE-M3 e conferindo cache de artigos...", flush=True)
            diagnostico_indice = busca_corpus.preparar_indice()
            AMBIENTE["buscador_local"] = diagnostico_indice
            print(json.dumps(diagnostico_indice, ensure_ascii=False, indent=2))
        sistema = SistemaMultiagentes(cliente, busca_corpus, config=CONFIG)
        identificador = datetime.now(timezone.utc).strftime("kaggle-" + MODO_BUSCA + "-%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
        ESTADO = sistema.executar(PERGUNTA, modo=MODO_BUSCA, run_id=identificador)
        ESTADO["ambiente_kaggle"] = dict(AMBIENTE)
        DESTINO_EXECUCAO = salvar_execucao(
            ESTADO, DIRETORIO_SAIDA, diagrama=sistema.grafo.get_graph().draw_mermaid(),
        )
        print("Estado:", ESTADO["status"], "| Motivo:", ESTADO.get("motivo"))
        print("Artefatos privados da execução:", DESTINO_EXECUCAO)
    except Exception as erro:
        ERRO_EXECUCAO = {
            "status": "falha_tecnica",
            "etapa": "inicializacao_ou_exportacao",
            "tipo": type(erro).__name__,
        }
        (DIRETORIO_SAIDA / "erro_execucao.json").write_text(
            json.dumps(ERRO_EXECUCAO, ensure_ascii=False, indent=2), encoding="utf-8",
        )
        print(json.dumps(ERRO_EXECUCAO, ensure_ascii=False, indent=2))
        print("Confira o pacote e os logs do Ollama. Nenhum resultado é presumido como aprovado.")
    finally:
        busca_corpus.fechar()


In [ ]:
def mostrar_execucao(estado):
    if estado is None:
        print("Sem estado de atendimento disponível. Consulte os diagnósticos acima.")
        return
    print("EVENTOS")
    for evento in estado.get("eventos", []):
        detalhe = " | ".join(
            f"{chave}={evento[chave]}"
            for chave in ("decisao", "modelo", "status")
            if chave in evento
        )
        print(f'{evento.get("numero", 0):02d} {evento.get("agente")}: {evento.get("etapa")} {detalhe}')
    print("\nVERSÕES E FEEDBACK")
    for versao in estado.get("historico", []):
        print("\nVersão", versao["versao"])
        print("Rascunho do Analista:\n", versao["rascunho"])
        print("Candidato após controles:\n", versao["candidato"])
        print("Guardrails:", json.dumps(versao["guardrails"], ensure_ascii=False))
        print("Feedback do Validador:", json.dumps(versao.get("feedback", {}), ensure_ascii=False))
    print("\nRESULTADO")
    print("Status:", estado["status"], "| Motivo:", estado.get("motivo"))
    print("Chamadas LLM:", estado["chamadas_llm"], "| Revisões:", estado["revisoes_usadas"])
    print("Falha inserida para teste:", estado.get("falha_injetada", False))
    print("Resposta final:\n", estado["resposta_final"])


if EXECUTAR:
    mostrar_execucao(ESTADO)


## Opcional: demonstrar uma correção controlada com material fictício

A opção **EXECUTAR_DEMO_REVISAO = False** mantém esta execução desativada. Para habilitar, altere a configuração e execute novamente. A busca passa a usar material fictício e o modo é **sintetico**.

O mecanismo de teste troca uma citação do primeiro candidato por **[Fonte 99]**, preservando o rascunho original e identificando **falha_injetada=true**. O Analista e o Validador continuam sendo os modelos reais. A demonstração depende das decisões efetivamente produzidas; não há feedback ou aprovação pré-fabricados. Se o Analista declarar insuficiência antes da preparação do candidato, o caso termina e a falha não é inserida.

Esse erro controlado não é aplicado ao corpus de legislação e não mede a frequência natural de erros do sistema.


In [ ]:
if EXECUTAR and EXECUTAR_DEMO_REVISAO:
    busca_demo = BuscaSintetica("ferias")
    try:
        sistema_demo = SistemaMultiagentes(cliente, busca_demo, config=CONFIG)
        identificador_demo = (
            datetime.now(timezone.utc).strftime("kaggle-sintetico-%Y%m%d-%H%M%S-")
            + uuid.uuid4().hex[:6]
        )
        ESTADO_DEMO = sistema_demo.executar(
            "Como organizar a leitura de informações sobre férias de acordo com o material fictício fornecido?",
            modo="sintetico", caso="revisao_controlada", run_id=identificador_demo,
        )
        ESTADO_DEMO["ambiente_kaggle"] = dict(AMBIENTE)
        destino_demo = salvar_execucao(
            ESTADO_DEMO, DIRETORIO_SAIDA,
            diagrama=sistema_demo.grafo.get_graph().draw_mermaid(),
        )
        mostrar_execucao(ESTADO_DEMO)
        print("Artefatos da demonstração sintética:", destino_demo)
    finally:
        busca_demo.fechar()


## Baixar os resultados

Cada execução exporta **estado.json, eventos.jsonl, fontes.json, resposta.txt, grafo.mmd e manifesto.json**. O estado preserva versões, feedback e consumo observado; o manifesto registra hashes dos artefatos.

A célula abaixo reúne as execuções em **multiagentes_resultados.zip**. Baixe o ZIP pela área **Output** do Kaggle. Logs de instalação, servidor e download dos modelos ficam separados em **/kaggle/working**.

No modo local, o cache também é reunido em **buscador_local_cache.zip**. Ele fica separado dos resultados e pode ser restaurado no diretório de cache em outra sessão. Os artigos só deixam de ser reprocessados se a identidade da base e do modelo permanecer igual.

Mantenha notebook, dataset e resultados privados enquanto revisar o material. Antes de compartilhar, confira as fontes, as respostas e os dados presentes nas saídas. O status **aprovada** significa que os controles do protótipo permitiram a liberação; não é certificação jurídica.


In [ ]:
if EXECUTAR and DIRETORIO_SAIDA.exists():
    arquivo_resultados = shutil.make_archive(
        str(WORKING / "multiagentes_resultados"), "zip", root_dir=DIRETORIO_SAIDA,
    )
    print("Arquivo para baixar:", arquivo_resultados)
    print("A execução só é concluída quando os estados e artefatos forem conferidos.")
    if MODO_BUSCA == "local" and CACHE_EMBEDDINGS.is_dir():
        arquivo_cache = shutil.make_archive(
            str(WORKING / "buscador_local_cache"), "zip", root_dir=CACHE_EMBEDDINGS,
        )
        print("Cache de artigos para restaurar em outra sessão:", arquivo_cache)
